In [1]:
import osmnx as ox
import pandas as pd

In [2]:
places = [
    "Amherst, Massachusetts, USA",
    "Northampton, Massachusetts, USA",
    "Springfield, Massachusetts, USA",
    "Holyoke, Massachusetts, USA",
    "Greenfield, Massachusetts, USA",
    "Westfield, Massachusetts, USA",
]

G = ox.graph_from_place(places, network_type="drive")

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

Nodes: 3400
Edges: 8850


In [3]:
nodes_gdf, edges_gdf = ox.graph_to_gdfs(G)


print("\n── Node columns (tags) ──")
print(nodes_gdf.columns.tolist())

print("\n── Edge columns (tags) ──")
print(edges_gdf.columns.tolist())


── Node columns (tags) ──
['y', 'x', 'street_count', 'highway', 'ref', 'geometry']

── Edge columns (tags) ──
['osmid', 'highway', 'lanes', 'maxspeed', 'name', 'width', 'oneway', 'reversed', 'length', 'geometry', 'ref', 'junction', 'bridge', 'tunnel', 'access']


In [4]:
# Add maxspeed to all of the roads

# Massachusetts default speed limits by road type
highway_speed_defaults = {
    "motorway":       65,
    "trunk":          55,
    "primary":        45,
    "secondary":      40,
    "tertiary":       35,
    "residential":    25,
    "unclassified":   25,
    "service":        15,
    "motorway_link":  45,
    "trunk_link":     35,
    "primary_link":   35,
    "secondary_link": 30,
}

def impute_maxspeed(row):
    speed = row["maxspeed"]
    highway = row["highway"]
    
    # flatten lists for both fields
    if isinstance(speed, list):
        speed = next((s for s in speed if s is not None), None)
    if isinstance(highway, list):
        highway = highway[0]
    
    if speed is not None and pd.notna(speed):
        # clean formats like "35 mph", "35", "MA:urban", "signals"
        cleaned = str(speed).replace(" mph", "").strip()
        numeric = pd.to_numeric(cleaned, errors="coerce")
        if pd.notna(numeric):
            return numeric
        # handle conditional OSM tags
        fallback_map = {
            "MA:urban":   25,
            "MA:rural":   50,
            "signals":    25,
            "walk":       10,
        }
        return fallback_map.get(cleaned, highway_speed_defaults.get(highway, 25))
    
    return highway_speed_defaults.get(highway, 25)

edges_gdf["maxspeed_imputed"] = edges_gdf.apply(impute_maxspeed, axis=1)
print(edges_gdf["maxspeed_imputed"].describe())

count    8850.000000
mean       29.349718
std         7.123077
min        10.000000
25%        25.000000
50%        25.000000
75%        35.000000
max        65.000000
Name: maxspeed_imputed, dtype: float64


In [5]:
print("\n── Edge 'highway' distribution ──")
print(edges_gdf["highway"].value_counts())

print("\n── Top edge names ──")
print(edges_gdf["name"].value_counts().head(20))


── Edge 'highway' distribution ──
highway
residential                    5457
tertiary                       1018
secondary                      1003
primary                         643
trunk                           405
unclassified                    157
motorway_link                    88
motorway                         33
secondary_link                   17
primary_link                     11
tertiary_link                     7
trunk_link                        5
[primary, trunk]                  2
[unclassified, tertiary]          2
[unclassified, residential]       2
Name: count, dtype: int64

── Top edge names ──
name
Main Street           139
Northampton Street    108
Elm Street             89
Western Avenue         76
Southampton Road       75
Pleasant Street        75
Franklin Street        64
South Street           63
Prospect Street        60
Homestead Avenue       58
Hampden Street         51
King Street            50
Meadow Street          47
Sargeant Street        46


In [6]:
sample_node_id = list(G.nodes)[0]
node_data      = G.nodes[sample_node_id]          # plain dict of all tags
print(f"\nSample node {sample_node_id}:")
print(node_data)


Sample node 71827554:
{'y': 42.09592, 'x': -72.7762672, 'street_count': 3}


In [7]:
sample_edge = list(G.edges(keys=True))[0]          # (u, v, key)
u, v, k     = sample_edge
edge_data   = G[u][v][k]                           # plain dict of all tags
print(f"\nSample edge ({u} → {v}, key={k}):")
print(edge_data)


Sample edge (71827554 → 71884443, key=0):
{'osmid': 9431473, 'highway': 'residential', 'lanes': '2', 'maxspeed': '35 mph', 'name': 'Plantation Circle', 'width': '15.2', 'oneway': False, 'reversed': True, 'length': np.float64(156.56112818848325)}


In [8]:
# ── 6. Build a lookup: node_id → tag dict  (handy for your pipeline) ─────────
node_tags = dict(G.nodes(data=True))

In [9]:
# ── 7. Build a lookup: (u, v, key) → tag dict ────────────────────────────────
edge_tags = {(u, v, k): data for u, v, k, data in G.edges(keys=True, data=True)}

In [10]:
# ── 8. Collect every unique tag key that appears across all edges/nodes ───────
all_node_tag_keys = set()
for _, data in G.nodes(data=True):
    all_node_tag_keys.update(data.keys())

all_edge_tag_keys = set()
for _, _, data in G.edges(data=True):
    all_edge_tag_keys.update(data.keys())

print("\nAll node tag keys:", all_node_tag_keys)
print("All edge tag keys:", all_edge_tag_keys)


All node tag keys: {'y', 'highway', 'street_count', 'x', 'ref'}
All edge tag keys: {'length', 'tunnel', 'osmid', 'reversed', 'oneway', 'lanes', 'bridge', 'highway', 'geometry', 'junction', 'maxspeed', 'access', 'name', 'width', 'ref'}


In [11]:
# ── 9. Filter edges/nodes by a specific tag ───────────────────────────────────
# e.g. all highway edges
highway_edges = edges_gdf[edges_gdf["highway"].isin(["motorway", "trunk", "primary"])]
print(f"\nMajor highway edges: {len(highway_edges)}")

# e.g. all nodes that have an 'amenity' tag (rare in drive graphs, but possible)
if "amenity" in nodes_gdf.columns:
    amenity_nodes = nodes_gdf[nodes_gdf["amenity"].notna()]
    print(f"Nodes with amenity tag: {len(amenity_nodes)}")


Major highway edges: 1081


In [12]:
# Continuous tags — get actual min/max/distribution
continuous_tags = ["maxspeed_imputed", "lanes", "length", "width"]
for tag in continuous_tags:
    if tag in edges_gdf.columns:
        col = pd.to_numeric(edges_gdf[tag], errors="coerce").dropna()
        print(f"{tag}: min={col.min()}, max={col.max()}, mean={col.mean():.1f}, median={col.median()}")

# Discrete tags — get the actual value set
discrete_tags = ["highway", "access", "bridge", "junction", "ref"]
for tag in discrete_tags:
    if tag in edges_gdf.columns:
        # flatten lists if needed
        vals = edges_gdf[tag].dropna().explode().unique()
        print(f"{tag}: {vals.tolist()}")

maxspeed_imputed: min=10, max=65, mean=29.3, median=25.0
lanes: min=1.0, max=5.0, mean=2.0, median=2.0
length: min=1.9604163159560468, max=6845.567569898566, mean=189.4, median=117.60476901675995
width: min=3.0, max=122.0, mean=14.7, median=14.6
highway: ['residential', 'trunk', 'secondary', 'tertiary', 'primary', 'motorway_link', 'unclassified', 'secondary_link', 'motorway', 'tertiary_link', 'primary_link', 'trunk_link']
access: ['yes', 'no']
bridge: ['yes', 'viaduct']
junction: ['roundabout', 'jughandle']
ref: ['US 20', 'US 5', 'MA 9', 'MA 10', 'MA 66', 'MA 187', 'US 202', 'US 20;US 202;MA 10', 'US 202;MA 10', 'I 91', 'MA 9;MA 10', 'US 5;MA 10', 'I 391', 'US 5;US 202', 'MA 141', 'I 90', 'MA 116;MA 141', 'MA 116']


In [13]:
tag_schema = {
    # continuous: sample uniformly or from percentiles of real data
    "continuous": {
        "maxspeed_imputed": {"min": 10, "max": 65, "mean": 29.4, "median": 25, "unit": "mph"},
        "lanes":    {"min": 1,  "max": 6, "mean": 2, "median": 2, "unit": "integer"},
        "length":   {"min": 1.9604163159560468,  "max": 6845.567569898566, "mean": 189.4, "median": 117.60476901675995, "unit": "meters"},
        "width":    {"min": 3,  "max": 122, "mean": 14.7, "median": 14.6, "unit": "meters"},
    },
    # discrete: only values that actually appear in your graph
    "discrete": {
        "highway":  ['residential', 'trunk', 'secondary', 'tertiary', 'primary', 'motorway_link', 
                     'unclassified', 'secondary_link', 'motorway', 'tertiary_link', 'primary_link', 'trunk_link'],
        "access":   ["yes", "no"],
        "bridge":   ["yes", "viaduct"],
        "junction": ["roundabout", "jughandle"],
        "ref":      ['US 20', 'US 5', 'MA 9', 'MA 10', 'MA 66', 'MA 187', 'US 202', 'US 20;US 202;MA 10', 'US 202;MA 10', 
                     'I 91', 'MA 9;MA 10', 'US 5;MA 10', 'I 391', 'US 5;US 202', 'MA 141', 'I 90', 'MA 116;MA 141', 'MA 116'],  # real refs from your graph
    }
}


In [14]:
import random
import json

def sample_route_context(tag_schema, n_constraints=2):
    context = {}
    
    # pick n_constraints tags to feature in this example
    all_tags = list(tag_schema["continuous"].keys()) + list(tag_schema["discrete"].keys())
    chosen = random.sample(all_tags, k=n_constraints)
    
    for tag in chosen:
        if tag in tag_schema["continuous"]:
            spec = tag_schema["continuous"][tag]
            val = random.uniform(spec["min"], spec["max"])
            context[tag] = {"value": round(val, 1), "unit": spec["unit"]}
        else:
            context[tag] = random.choice(tag_schema["discrete"][tag])
    
    return context

# Example output:
# {"highway": "motorway", "maxspeed": {"value": 28.3, "unit": "mph"}}

sample_route_context(tag_schema, n_constraints=3)

{'highway': 'primary_link',
 'length': {'value': 5477.0, 'unit': 'meters'},
 'access': 'yes'}

In [15]:
SYSTEM_PROMPT = """You generate realistic navigation requests for the Pioneer Valley 
region of Massachusetts (Amherst, Northampton, Springfield, Holyoke, Greenfield, Westfield).

Given a set of route constraints expressed as graph tags and values, write a natural 
language navigation prompt a real user might type. 

Rules:
- The prompt must implicitly or explicitly encode all provided constraints
- Vary phrasing significantly across examples — don't repeat templates
- Include a plausible start and end location from the Pioneer Valley
- Keep it to 1-2 sentences
- Do NOT mention tag names directly (don't say "highway tag"). Use natural language.
- Output JSON only: {"start": "...", "end": "...", "prompt": "...", "constraints": {...}}
"""

def build_user_message(context):
    return f"Generate a navigation prompt encoding these constraints: {json.dumps(context)}"

In [38]:
from dotenv import load_dotenv
import os
import json
import random
import time
from cerebras.cloud.sdk import Cerebras

load_dotenv()

client = Cerebras(
    api_key=os.environ.get("CEREBRAS_API_KEY")
)

tag_schema = {
    "continuous": {
        "maxspeed_imputed": {"min": 10, "max": 65, "mean": 29.4, "median": 25, "unit": "mph"},
        "lanes":    {"min": 1,  "max": 6, "mean": 2, "median": 2, "unit": "integer"},
        "length":   {"min": 1.96,  "max": 6845.57, "mean": 189.4, "median": 117.60, "unit": "meters"},
        "width":    {"min": 3,  "max": 122, "mean": 14.7, "median": 14.6, "unit": "meters"},
    },
    "discrete": {
        "highway":  ['residential', 'trunk', 'secondary', 'tertiary', 'primary', 'motorway_link', 
                     'unclassified', 'secondary_link', 'motorway', 'tertiary_link', 'primary_link', 'trunk_link'],
        "access":   ["yes", "no"],
        "bridge":   ["yes", "viaduct"],
        "junction": ["roundabout", "jughandle"],
        "ref":      ['US 20', 'US 5', 'MA 9', 'MA 10', 'MA 66', 'MA 187', 'US 202', 'US 20;US 202;MA 10', 'US 202;MA 10', 
                     'I 91', 'MA 9;MA 10', 'US 5;MA 10', 'I 391', 'US 5;US 202', 'MA 141', 'I 90', 'MA 116;MA 141', 'MA 116'],
    }
}

SYSTEM_PROMPT = """You generate realistic navigation requests for the Pioneer Valley 
region of Massachusetts (Amherst, Northampton, Springfield, Holyoke, Greenfield, Westfield).

You will be given a list of constraint sets derived from road network tags. For each one, 
generate a natural language navigation prompt that a real user might type or say.

The constraints come from the following tag schema:
- maxspeed_imputed: speed limit in float mph (min=10, max=65, mean=29.4, median=25). Higher values indicate faster, more major roads.
- lanes: integer number of lanes (min=1, max=6, mean=2, median=2). More lanes = busier, more major road.
- length: float edge length in meters (min=1.96, max=6845.57, mean=189.4, median=117.6). Longer edges = more highway-like.
- width: float road width in meters (min=3, max=122, mean=14.7, median=14.6). Wider = more major road.
- highway: road classification. From most to least major: motorway > trunk > primary > secondary > tertiary > unclassified > residential. *_link variants are on/off ramps.
- access: whether the road is publicly accessible. "yes" = public, "no" = restricted.
- bridge: whether the road is a bridge ("yes") or viaduct.
- junction: type of junction. "roundabout" = circular junction, "jughandle" = indirect left turn.
- ref: route reference number. I-prefixed = interstate highway, US-prefixed = US route, MA-prefixed = Massachusetts state route.

Rules:
- Each prompt must implicitly or explicitly encode all constraints in its set, but naturally — as a real user would phrase them
- Do NOT mention tag names directly (never say "highway tag", "maxspeed", "ref", etc.)
- Vary phrasing significantly across examples — do not reuse sentence templates
- Include a plausible start and end location within the Pioneer Valley for each example
- Keep each prompt to 1-2 sentences
- Use real Pioneer Valley landmarks, towns, streets, and routes where possible
- Output a JSON array only — no preamble, no markdown backticks, no explanation
- Have a good mix of phrases where the location is first and when the constraints are first

Each element must follow this exact format:
{"id": <int>, "start": "...", "end": "...", "prompt": "...", "constraints": {...}}

Example input constraint set:
{"id": 0, "constraints": {"highway": "motorway", "ref": "I 91", "maxspeed_imputed": {"value": 65, "unit": "mph"}}}

Example output:
{"id": 0, "start": "Northampton, MA", "end": "Springfield, MA", "prompt": "Take me from Northampton to Springfield as fast as possible — I don't mind using the interstate.", "constraints": {"highway": "motorway", "ref": "I 91", "maxspeed_imputed": {"value": 65, "unit": "mph"}}}"""

def sample_route_context(tag_schema, n_constraints=4):
    context = {}
    all_tags = list(tag_schema["continuous"].keys()) + list(tag_schema["discrete"].keys())
    chosen = random.sample(all_tags, k=min(n_constraints, len(all_tags)))
    
    for tag in chosen:
        if tag in tag_schema["continuous"]:
            spec = tag_schema["continuous"][tag]
            val = random.uniform(spec["min"], spec["max"])
            context[tag] = {"value": round(val, 1), "unit": spec["unit"]}
        else:
            context[tag] = random.choice(tag_schema["discrete"][tag])
    
    return context

def strip_markdown(raw):
    """Remove ```json ... ``` wrapping if present."""
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return raw.strip()

def repair_truncated_json(raw):
    """Try to salvage partial JSON by truncating at the last complete object."""
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        # find the last complete object — last occurrence of "}," or "}\n"
        last_complete = max(raw.rfind("},"), raw.rfind("}\n"))
        if last_complete == -1:
            return []
        truncated = raw[:last_complete + 1] + "\n]"
        try:
            return json.loads(truncated)
        except json.JSONDecodeError:
            return []

def generate_dataset_batched(n_examples, tag_schema, batch_size=50, output_path="synthetic_dataset.jsonl"):
    all_contexts = []
    for i in range(n_examples):
        n_constraints = random.choices([1, 2, 3], weights=[0.3, 0.5, 0.2])[0]
        context = sample_route_context(tag_schema, n_constraints=n_constraints)
        all_contexts.append({"id": i, "constraints": context})
    
    all_results = []
    batches = [all_contexts[i:i+batch_size] for i in range(0, n_examples, batch_size)]
    
    with open(output_path, "a") as f:
        for batch_num, batch in enumerate(batches):
            print(f"Batch {batch_num+1}/{len(batches)} ({len(batch)} examples)...")
            
            user_message = f"""Generate a navigation prompt for each of the following {len(batch)} constraint sets.

{json.dumps(batch, indent=2)}"""
            
            raw = ""
            try:
                response = client.chat.completions.create(
                    model="llama3.1-8b",
                    max_completion_tokens=8096,
                    temperature=0.2,
                    top_p=1,
                    stream=False,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": user_message}
                    ]
                )
                raw = strip_markdown(response.choices[0].message.content.strip())
                parsed = repair_truncated_json(raw)
                
                if len(parsed) == 0:
                    print(f"  ✗ Batch {batch_num+1} could not be recovered, skipping")
                    continue
                
                if len(parsed) != len(batch):
                    print(f"  ⚠ Recovered {len(parsed)}/{len(batch)} examples from truncated output")
                
                context_lookup = {c["id"]: c["constraints"] for c in batch}
                for item in parsed:
                    item["sampled_context"] = context_lookup.get(item["id"], {})
                    f.write(json.dumps(item) + "\n")
                    all_results.append(item)
                
                print(f"  ✓ {len(parsed)} examples written")
                time.sleep(0.2)
                
            except json.JSONDecodeError as e:
                print(f"  ✗ Batch {batch_num+1} unrecoverable JSON error: {e}")
                print(f"  Raw output snippet: {raw[:300]}...")
            except Exception as e:
                print(f"  ✗ Batch {batch_num+1} failed: {e}")
    
    print(f"\nDone: {len(all_results)}/{n_examples} examples written to {output_path}")
    return all_results

dataset = generate_dataset_batched(2000, tag_schema, batch_size=50, output_path="synthetic_dataset.jsonl")

Batch 1/40 (50 examples)...
  ✓ 50 examples written
Batch 2/40 (50 examples)...
  ✓ 50 examples written
Batch 3/40 (50 examples)...
  ✓ 50 examples written
Batch 4/40 (50 examples)...
  ✓ 50 examples written
Batch 5/40 (50 examples)...
  ✓ 50 examples written
Batch 6/40 (50 examples)...
  ✓ 50 examples written
Batch 7/40 (50 examples)...
  ✓ 50 examples written
Batch 8/40 (50 examples)...
  ✓ 50 examples written
Batch 9/40 (50 examples)...
  ✓ 50 examples written
Batch 10/40 (50 examples)...
  ✓ 50 examples written
Batch 11/40 (50 examples)...
  ✓ 50 examples written
Batch 12/40 (50 examples)...
  ✓ 50 examples written
Batch 13/40 (50 examples)...
  ⚠ Recovered 49/50 examples from truncated output
  ✓ 49 examples written
Batch 14/40 (50 examples)...
  ✗ Batch 14 could not be recovered, skipping
Batch 15/40 (50 examples)...
  ✗ Batch 15 could not be recovered, skipping
Batch 16/40 (50 examples)...
  ✓ 50 examples written
Batch 17/40 (50 examples)...
  ✓ 50 examples written
Batch 18/40 (

In [54]:
# Deduplicate Code
seen_prompts = set()
duplicates = 0

with open("synthetic_dataset.jsonl", "r") as fin:
    lines = fin.readlines()

with open("synthetic_dataset.jsonl", "w") as fout:
    for line in lines:
        item = json.loads(line)
        prompt = item["prompt"]
        if prompt not in seen_prompts:
            seen_prompts.add(prompt)
            fout.write(line)
        else:
            duplicates += 1

print(f"Removed {duplicates} duplicates, {len(seen_prompts)} unique examples remain")


# Fix Id Numbers
with open("synthetic_dataset.jsonl", "r") as fin:
    lines = fin.readlines()

with open("synthetic_dataset.jsonl", "w") as fout:
    for new_id, line in enumerate(lines):
        item = json.loads(line)
        item["id"] = new_id
        fout.write(json.dumps(item) + "\n")

print(f"Reassigned IDs 0 to {new_id}")

Removed 0 duplicates, 5754 unique examples remain
Reassigned IDs 0 to 5753
